In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler

plt.style.use('dark_background')
np.random.seed(42)
torch.manual_seed(42)

print("✅ Ready!")


✅ Ready!


In [3]:
# Load dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

print(f"Dataset: Breast Cancer Wisconsin")
print(f"Total samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Classes: {len(np.unique(y))} (binary)")


Dataset: Breast Cancer Wisconsin
Total samples: 569
Features: 30
Classes: 2 (binary)


In [4]:
# Split: Training+Validation (80%) vs Test (20%)
'''
We will use the function: train_test_split
This function takes the following inputs:
X
The full feature matrix (all input variables).
Shape: (n_samples, n_features).

y
The full target vector (labels or regression targets).
Shape: (n_samples,).

test_size=0.2
Proportion of the dataset to allocate to the test set.
Here: 20% test, 80% train/validation.

random_state=42
Seed for the random number generator.
Ensures reproducibility of the split.

stratify=y
Ensures the class distribution in y is preserved in both train and test sets.
Important for classification tasks with imbalanced classes.


Outputs:
X_train_val → 80% of the feature matrix (training/validation set).

X_test → 20% of the feature matrix (test set).

y_train_val → Corresponding labels for the training/validation set.

y_test → Corresponding labels for the test set.
'''
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)  # note that this is a single split and not a cross-validation

print(f"\nData split:")
print(f"  Training+Validation: {X_train_val.shape[0]} samples")
print(f"  External Test Set:   {X_test.shape[0]} samples")
print(f"\nTest set will NOT be used during hyperparameter tuning!")



Data split:
  Training+Validation: 455 samples
  External Test Set:   114 samples

Test set will NOT be used during hyperparameter tuning!


In [6]:
class SimpleNN(nn.Module):
    def __init__(self, input_size=30, hidden_size=15, num_classes=2):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Create model
model = SimpleNN()
print("Architecture:")
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")


Architecture:
SimpleNN(
  (fc1): Linear(in_features=30, out_features=15, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=15, out_features=2, bias=True)
)

Parameters: 497


In [7]:
def train_model(model, X_train, y_train, X_val, y_val, lr, epochs=100):
    """
    Train model and return learning & validation curves

    Returns:
        train_losses, val_losses, train_accs, val_accs
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []

    for epoch in range(epochs):
        # Training
        model.train()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, train_pred = torch.max(outputs, 1)
        train_acc = (train_pred == y_train).float().mean().item()
        train_losses.append(loss.item())
        train_accs.append(train_acc)

        # Validation
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val)
            val_loss = criterion(val_outputs, y_val)
            _, val_pred = torch.max(val_outputs, 1)
            val_acc = (val_pred == y_val).float().mean().item()

        val_losses.append(val_loss.item())
        val_accs.append(val_acc)

    return train_losses, val_losses, train_accs, val_accs

print("Training function defined")


Training function defined


## Exercise: Repeat with 10-Fold CV


In [8]:
# YOUR TASK: Perform 10-fold CV for LR=0.001
test_lr = 0.001

# Initialize 10-Fold Cross-Validation
kfold_10 = KFold(n_splits=10, shuffle=True, random_state=42)
fold_accs_10 = []

# Loop over all 10 splits
for train_idx, val_idx in kfold_10.split(X_train_val):
    # 1. Normalize fold data using training fold statistics only
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train_val[train_idx])
    X_va = scaler.transform(X_train_val[val_idx])

    # 2. Convert NumPy arrays to PyTorch tensors
    X_tr_t = torch.FloatTensor(X_tr)
    y_tr_t = torch.LongTensor(y_train_val[train_idx])
    X_va_t = torch.FloatTensor(X_va)
    y_va_t = torch.LongTensor(y_train_val[val_idx])

    # 3. Create a fresh model and train it for 100 epochs with lr=test_lr (0.001)
    model_10 = SimpleNN()
    _, _, _, val_accs = train_model(
        model_10, X_tr_t, y_tr_t, X_va_t, y_va_t, lr=test_lr, epochs=100
    )

    # 4. Save the final epoch's validation accuracy for this fold
    fold_accs_10.append(val_accs[-1])

# Calculate mean and standard deviation across the 10 folds
mean_10fold = np.mean(fold_accs_10)
std_10fold = np.std(fold_accs_10)

print(f"\n10-Fold CV Results (LR={test_lr}):")
print(f"  Mean Accuracy: {mean_10fold:.4f} ± {std_10fold:.4f}")



10-Fold CV Results (LR=0.001):
  Mean Accuracy: 0.9626 ± 0.0358
